# 10 — Distributed Training: DDP and FSDP Templates

Goal: scale training across GPUs with standard templates.

_Generated: 2026-01-25_

## Setup

```bash
pip install torch torchvision torchaudio
pip install transformers datasets tokenizers accelerate evaluate
pip install matplotlib tensorboard
```

In [ ]:

import os, math, random
import numpy as np
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("torch:", torch.__version__)
print("device:", device)

## 1. DDP template (torchrun)

Launch:
```bash
torchrun --nproc_per_node=NUM_GPUS train_ddp.py
```

In [ ]:

ddp_script = r'''
# train_ddp.py
import os
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler, TensorDataset

def main():
    dist.init_process_group(backend="nccl")  # or "gloo"
    local_rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    X = torch.randn(4096, 10)
    y = torch.randint(0, 2, (4096,))
    ds = TensorDataset(X, y)
    sampler = DistributedSampler(ds, shuffle=True)
    dl = DataLoader(ds, batch_size=128, sampler=sampler, num_workers=2, pin_memory=True)

    model = nn.Sequential(nn.Linear(10, 64), nn.ReLU(), nn.Linear(64, 2)).to(device)
    model = DDP(model, device_ids=[local_rank])
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(5):
        sampler.set_epoch(epoch)
        for xb, yb in dl:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            loss = loss_fn(model(xb), yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

    dist.destroy_process_group()

if __name__ == "__main__":
    main()
'''
print(ddp_script[:900] + "\n...\n")

## 2. FSDP sketch

FSDP shards parameters/gradients/optimizer state for very large models.
Checkpointing and wrapping policy matter.

In [ ]:

fsdp_script = r'''
# train_fsdp.py (sketch)
import os
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp.wrap import size_based_auto_wrap_policy

def main():
    dist.init_process_group(backend="nccl")
    local_rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    model = nn.Sequential(nn.Linear(4096, 4096), nn.GELU(), nn.Linear(4096, 4096)).to(device)
    wrap_policy = size_based_auto_wrap_policy(min_num_params=int(1e8))
    model = FSDP(model, auto_wrap_policy=wrap_policy)
    # training loop similar to DDP (optimizer, loss, backward, step)
    # checkpointing requires FSDP state_dict utilities.

if __name__ == "__main__":
    main()
'''
print(fsdp_script[:900] + "\n...\n")